# Construction of Jacobian based P-V and V-Q curves
This notebook is aiming to indtroduce the construction of $P-V$ and $V-Q$ curves. Analytical calculation of stability phenomena helps discussing and evaluating the performance of indices.

## Imports and Predefinitions

In [ ]:
# %matplotlib widget
# import ipympl

import os
import sys
import numpy as np
import pandas as pd

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler")
sys.path.insert(1, parent)
sys.path.append(
    str(os.path.dirname(os.path.dirname(os.path.abspath("p-v_curves.ipynb"))))
)

import matplotlib.pyplot as plt
import matplotlib as mpl
from tools import *

save = os.path.join(os.getcwd(), "plots/")

from src.diffpssi.stability_lib.voltage import NoseCurve

color = [ees_blue, ees_yellow, ees_green, ees_red, ees_lightblue]

## Simple cases: A source - load model

With a simple two-bus system, just a load $P+jQ$ and a transmission impedance $jX$, and using the load flow equations
$$\begin{align}
P&=-\frac{EV}{X}\sin\vartheta \text{, and }\\
Q&=-\frac{V^2}{X}+\frac{EV}{X}\cos\vartheta \text{,}
\end{align}$$
one can obtain a from $V$ dependent function
$$\begin{align}
V=...
\end{align}$$

### 2D curves

In [ ]:
def p_q_v_upper(E, Q, P, X):
    return np.sqrt(
        (E**2) / 2 - Q * X + np.sqrt((E**4) / 4 - (X**2) * (P**2) - X * (E**2) * Q)
    )


def p_q_v_lower(E, Q, P, X):
    return np.sqrt(
        (E**2) / 2 - Q * X - np.sqrt((E**4) / 4 - (X**2) * (P**2) - X * (E**2) * Q)
    )


def p_v(E, V, X):
    return V / X * np.sqrt(E**2 - V**2)


def q_v(E, V, X):
    return V / X * np.sqrt(E**2 - V**2)


def p_vq(E, V, Q, X):
    p = 1 / X * np.sqrt(-(V**4) - (2 * Q * X - E**2) * V**2 - X**2 * Q**2)
    return p

In [ ]:
# reactance for only the transformer related on the system parameters
X = 0.15 / 2200

# Voltage at slack node = 1
E = 1  # np.linspace(0, 1, 10)
V = np.linspace(0, 1, 1000)

P = p_v(E, V, X)

In [ ]:
plt.figure(figsize=(5, 4))
plt.plot(P * X / E**2, V / E, label=r"$\tan \phi=0$")
plt.legend()
plt.xlabel(r"Active Power $\frac{PX}{E^2}$ in p.u.")
plt.ylabel(r"Voltage $\frac{V}{E}$ in p.u.")
plt.grid()
plt.savefig(save + "p_v_analytical.pdf")
plt.show()

In [ ]:
p = np.linspace(0, 10000, 1000)
q = 0 * p
x = X * np.ones_like(p)
e = np.ones_like(p)

v_1_upper = p_q_v_upper(E, q, p, X)
v_1_lower = p_q_v_lower(E, q, p, X)

In [ ]:
plt.figure(figsize=(5, 4))
plt.plot(p, v_1_upper, label=r"$\tan \phi=0$", color=ees_blue)
plt.plot(p, v_1_lower, label=r"$\tan \phi=0$", color=ees_blue)
plt.legend()
plt.xlabel("Active Power")
plt.ylabel("Voltage")
plt.grid()
plt.show()

In [ ]:
# tan_phi = [-0.4, -0.2, 0, 0.2, 0.4]

# v_2_upper = []
# v_2_lower = []

# for phi in tan_phi:
#     q = phi * p
#     v_2_upper.append(p_q_v_upper(E, q, p, X))
#     v_2_lower.append(p_q_v_lower(E, q, p, X))

In [ ]:
# plt.figure(figsize=(12,8))

# for i, phi in enumerate(tan_phi):
#     plt.plot(p, v_2_upper[i], label=f'$\tan \phi={phi}$', color=color[i])
#     plt.plot(p, v_2_lower[i], color=color[i])

# plt.legend()
# plt.xlabel('Active Power')
# plt.ylabel('Voltage')

# plt.grid()
# plt.show()

### Q-V Curves

In [ ]:
# p_qc = 0.5 * E**2 / X


# def q_c(V, E, X, cos_phi):
#     return - V**2 / X + E*V/X * cos_phi

# # V = np.linspace(0, 1.3, 100)#

# q_curve = q_c(V, E, X, cos_phi=1)

In [ ]:
# plt.figure(figsize=(5,4))
# plt.plot(V/E, q_curve*X/E**2, label=r'$\cos \phi=1$')
# plt.legend()
# plt.ylabel(r'Reactive Power $\frac{Q_\mathrm{r}X}{E^2}$ in p.u.')
# plt.xlabel(r'Voltage $\frac{V}{E}$ in p.u.')
# plt.grid()
# plt.savefig(save + 'v_q_analytical.pdf')
# plt.show()

### 3D curves

## VQ Curves

## Comparison Analytical vs. Numerical solution

Define the test grid first. This is a simle generator - load model with an simple transformer in between.

In [ ]:
def test_system(trans_type="simple", control="oltc", B1=[400, 0], param_dict_oltc=None):
    return {
        "base_mva": 2200,
        "f": 60,
        "slack_bus": "B0",
        "base_voltage": 100,
        "busses": [
            ["name", "V_n"],
            ["B0", 10],
            ["B1", 100],
        ],
        "transformers": [
            [
                "type",
                "control",
                "name",
                "from_bus",
                "to_bus",
                "S_n",
                "tap_side",
                "measure_side",
                "V_n_from",
                "V_n_to",
                "R",
                "X",
                "param_dict_oltc",
            ],
            [
                trans_type,
                control,
                "T1",
                "B0",
                "B1",
                2200,
                "hv",
                "hv",
                10,
                100,
                0,
                0.15,
                param_dict_oltc,
            ],
        ],
        "generators": {
            "GEN": [
                [
                    "name",
                    "bus",
                    "S_n",
                    "V_n",
                    "P",
                    "V",
                    "H",
                    "D",
                    "X_d",
                    "X_q",
                    "X_d_t",
                    "X_q_t",
                    "X_d_st",
                    "X_q_st",
                    "T_d0_t",
                    "T_q0_t",
                    "T_d0_st",
                    "T_q0_st",
                ],
                [
                    "G1",
                    "B0",
                    2200,
                    10,
                    -1998,
                    1,
                    3.5,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
            ],
        },
        "loads": {
            "ZIP": [
                ["name", "bus", "P", "Q", "model"],
                ["L1", "B1", B1[0], B1[1], "Z"],
            ],
        },
    }

In [ ]:
p_load = np.linspace(0, 10000, 1000)
# tan_phi = np.linspace(-0.2, 5, 8)
tan_phi = [0, 1]

nose_curve = NoseCurve(
    load_model=test_system,
    loading={"p": p_load, "tan_phi": tan_phi},
)

result_mesh = nose_curve.run_calculation(bus=["B1"])["B1"]

In [ ]:
plt.figure(figsize=(8, 5))

# Add theoretical curves
plt.plot(P, V, label=r"Analytical - $\tan(\phi)=0$")

for phi in [0]:
    plt.plot(
        result_mesh[result_mesh["tan_phi"] == phi]["p"],
        np.abs(result_mesh[result_mesh["tan_phi"] == phi]["v"]),
        label=r"Numerical - $\tan(\phi)={}$".format(phi),
    )

plt.xlabel("Active Power in MW")
plt.ylabel("Voltage in p.u.")

plt.legend()
plt.grid()

plt.savefig(save + f"simple_load_B1_nose_curve_w-theoretical.pdf")
plt.show()